In [0]:

def transform_inpatient(df):
    return df.dropDuplicates(['ClaimID'])

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, concat_ws
def validate_inpatinet(df):
    invalid_condition = (
        (col('ClaimID').isNull()) |
        (col("BeneID").isNull()) |
        (col("Provider").isNull()) |
        (col("AttendingPhysician").isNull()) |
        (col("DiagnosisGroupCode").isNull()) |
        (col("ClmDiagnosisCode_1").isNull()) |
        (col("ClaimStartDt") > col("ClaimEndDt")) |
        (col("AdmissionDt") > col("DischargeDt")) |
        (col("AdmissionDt") < col("ClaimStartDt")) |
        (col("DischargeDt") > col("ClaimEndDt"))
    )

    valid_df = df.filter(~invalid_condition)
    reject_df = df.filter(invalid_condition)\
        .withColumn(
            "reject_reason",
            concat_ws(
                "; ",
                when(col("ClaimID").isNull(), "ClaimID is NULL"),
                when(col("BeneID").isNull(), "BeneID is NULL"),
                when(col("Provider").isNull(), "Provider is NULL"),
                when(col("AttendingPhysician").isNull(), "Attending Physician is NULL"),
                when(col("DiagnosisGroupCode").isNull(), "DiagnosisGroupCode is NULL"),
                when(col("ClmDiagnosisCode_1").isNull(), "Primary Diagnosis Missing"),
                when(col("ClaimStartDt") > col("ClaimEndDt"), "Claim Start Date after Claim End Date"),
                when(col("AdmissionDt") > col("DischargeDt"), "Admission Date after Discharge Date"),
                when(col("AdmissionDt") < col("ClaimStartDt"), "Admission before Claim Start"),
                when(col("DischargeDt") > col("ClaimEndDt"), "Discharge after Claim End")
            )
        )\
        .withColumn(
            "reject_ts",
            current_timestamp()
        )
    return valid_df, reject_df
# The error was caused by isNUll() instead of isNull()